# Hierarchical Indexing

Hierarchical / Multi-level indexing is very exciting as it opens the door to some quite sophisticated data analysis and manipulation, especially for working with higher dimensional data. In essence, it enables you to store and manipulate data with an arbitrary number of dimensions in lower dimensional data structures like Series (1d) and DataFrame (2d).

## Creating a MultiIndex (hierarchical index) object

The MultiIndex object is the hierarchical analogue of the standard Index object which typically stores the axis labels in pandas objects. You can think of MultiIndex as an array of tuples where each tuple is unique. A MultiIndex can be created from a list of arrays (using MultiIndex.from_arrays()), an array of tuples (using MultiIndex.from_tuples()), a crossed set of iterables (using MultiIndex.from_product()), or a DataFrame (using MultiIndex.from_frame()). The Index constructor will attempt to return a MultiIndex when it is passed a list of tuples. The following examples demonstrate different ways to initialize MultiIndexes.

In [4]:
import pandas as pd
import numpy as np

In [2]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]
arrays

[['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'],
 ['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two']]

In [6]:
arrays[0]

['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux']

In [3]:
tuples = list(zip(*arrays))
tuples1 = list(zip(arrays[0],arrays[1]))
print(f'{tuples == tuples1}\n')
tuples

True



[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [18]:
[list(set(x)) for x in iterables]

[['qux', 'baz', 'foo', 'bar'], ['one', 'two']]

In [8]:
languages = ['Java', 'Python', 'JavaScript']
versions = [14, 3, 6]

result = zip(languages, versions)
list(result)

[('Java', 14), ('Python', 3), ('JavaScript', 6)]

In [26]:
index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [10]:
s = pd.Series(np.random.randn(8), index=index)
s

first  second
bar    one      -0.330286
       two      -0.290773
baz    one       0.276299
       two      -0.480480
foo    one      -0.702578
       two       0.582828
qux    one       1.351767
       two      -0.172869
dtype: float64

paring every elements in two iterables (모든 경우의 수)

In [12]:
iterables = [["bar", "baz", "foo", "qux"], ["one", "two"]]

In [12]:
pd.MultiIndex.from_product(iterables, names=["first", "second"])

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [13]:
df = pd.DataFrame(
    [["bar", "one"], ["bar", "two"], ["foo", "one"], ["foo", "two"]],
    columns=["first", "second"],
)
df

,first,second
0,bar,one
1,bar,two
2,foo,one
3,foo,two


In [14]:
pd.MultiIndex.from_frame(df)

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('foo', 'one'),
            ('foo', 'two')],
           names=['first', 'second'])

As a convenience, you can pass a list of arrays directly into Series or DataFrame to construct a MultiIndex automatically:

In [15]:
arrays = [
    np.array(["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"]),
    np.array(["one", "two", "one", "two", "one", "two", "one", "two"]),
]
s = pd.Series(np.random.randn(8), index=arrays)
s

bar  one    0.340870
     two   -0.748559
baz  one    0.168850
     two   -1.106369
foo  one    0.154255
     two   -0.518614
qux  one    0.762024
     two   -0.858608
dtype: float64

## Reconstructing the level labels

In [27]:
index.get_level_values(1) # by ind

Index(['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two'], dtype='object', name='second')

In [28]:
index.get_level_values("first") # by specific name

Index(['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

## Basic indexing on axis with MultiIndex

One of the important features of hierarchical indexing is that you can select data by a “partial” label identifying a subgroup in the data. Partial selection “drops” levels of the hierarchical index in the result in a completely analogous way to selecting a column in a regular DataFrame:

In [30]:
df = pd.DataFrame(np.random.randn(3, 8), index=["A", "B", "C"], columns=index)
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A       0.002473  0.719473 -1.789992  0.459676  1.064197  0.404311  0.306028   
B       0.440120 -0.968734 -0.535919  3.041129  1.499826 -0.117679  0.241046   
C      -0.134473 -0.477325 -0.614198 -0.720573 -0.775154 -1.166960  0.150288   

first             
second       two  
A      -2.820494  
B      -1.695344  
C      -2.066111

In [61]:
df[['bar','baz']]

first        bar                 baz          
second       one       two       one       two
A       0.783184  1.163358 -0.062280  0.170354
B       0.102986 -0.496035 -0.095716  0.790188
C       0.448053 -0.837275 -0.703227 -0.790804

In [38]:
df["bar","one"]

A    0.783184
B    0.102986
C    0.448053
Name: (bar, one), dtype: float64

## Define levels

In [ ]:
df.columns.levels

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

In [63]:
df[["foo","qux"]].columns.levels  # same as original

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

You can use instead

In [24]:
df[["foo", "qux"]].columns.to_numpy()

array([('foo', 'one'), ('foo', 'two'), ('qux', 'one'), ('qux', 'two')],
      dtype=object)

In [25]:
df[["foo", "qux"]].columns.get_level_values(0)

Index(['foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

In [26]:
df[["foo", "qux"]].columns.get_level_values(1)

Index(['one', 'two', 'one', 'two'], dtype='object', name='second')

## Data alignment and using reindex

Operations between differently-indexed objects having MultiIndex on the axes will work as you expect; data alignment will work the same as an Index of tuples:

In [32]:
s

bar  one    0.340870
     two   -0.748559
baz  one    0.168850
     two   -1.106369
foo  one    0.154255
     two   -0.518614
qux  one    0.762024
     two   -0.858608
dtype: float64

In [70]:
s[:-2]

bar  one   -2.128385
     two   -0.530719
baz  one   -0.740476
     two   -0.774127
foo  one   -0.402026
     two   -0.953325
dtype: float64

In [72]:
s[3:]

baz  two   -0.774127
foo  one   -0.402026
     two   -0.953325
qux  one    1.782297
     two    1.209702
dtype: float64

In [36]:
s

bar  one    0.340870
     two   -0.748559
baz  one    0.168850
     two   -1.106369
foo  one    0.154255
     two   -0.518614
qux  one    0.762024
     two   -0.858608
dtype: float64

In [ ]:
s + s[:-2] # operating based on tuple(multi) indexes

bar  one    1.244312
     two   -2.887112
baz  one   -0.932175
     two   -3.587452
foo  one    1.319688
     two    1.412387
qux  one         NaN
     two         NaN
dtype: float64

In [ ]:
s[::2] # step

bar  one    0.622156
baz  one   -0.466087
foo  one    0.659844
qux  one   -0.730235
dtype: float64

In [31]:
s + s[::2]

bar  one    1.244312
     two         NaN
baz  one   -0.932175
     two         NaN
foo  one    1.319688
     two         NaN
qux  one   -1.460470
     two         NaN
dtype: float64

In [32]:
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [33]:
s[index[:3]]

bar  one    0.622156
     two   -1.443556
baz  one   -0.466087
dtype: float64

The reindex() method of Series/DataFrames can be called with another MultiIndex, or even a list or array of tuples:

In [54]:
np.random.randn(8).shape

(8,)

In [42]:
s

bar  one    0.340870
     two   -0.748559
baz  one    0.168850
     two   -1.106369
foo  one    0.154255
     two   -0.518614
qux  one    0.762024
     two   -0.858608
dtype: float64

In [69]:
s.index[-1]

('qux', 'two')

In [79]:
s.reindex(s.index[::-1])

qux  two   -0.858608
     one    0.762024
foo  two   -0.518614
     one    0.154255
baz  two   -1.106369
     one    0.168850
bar  two   -0.748559
     one    0.340870
dtype: float64

In [37]:
s.reindex(index[:3])

first  second
bar    one       0.340870
       two      -0.748559
baz    one       0.168850
dtype: float64

## Advanced indexing with hierarchical index

In [87]:
df

A         B         C
first second                              
bar   one    -0.151236 -0.537478 -1.178906
      two     1.607512 -1.057429 -2.191616
baz   one     0.256472 -0.154047 -0.844430
      two    -1.659504  1.078600 -0.934403
foo   one     0.473407 -0.097931 -0.842520
      two     0.064618  1.689765 -0.965431
qux   one    -0.142966 -1.426479  0.378445
      two     0.388319  0.287059 -1.216639

In [74]:
df.T

A         B         C
first second                              
bar   one     0.783184  0.102986  0.448053
      two     1.163358 -0.496035 -0.837275
baz   one    -0.062280 -0.095716 -0.703227
      two     0.170354  0.790188 -0.790804
foo   one    -0.582438  0.074955  2.109549
      two     0.064365  0.434265 -1.183725
qux   one     0.398612 -0.271885  1.034425
      two     1.507275 -0.855506  1.530736

In [68]:
df = df.T

In [77]:
df.index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [69]:
df[['A','B']]

A         B
first second                    
bar   one     0.002473  0.440120
      two     0.719473 -0.968734
baz   one    -1.789992 -0.535919
      two     0.459676  3.041129
foo   one     1.064197  1.499826
      two     0.404311 -0.117679
qux   one     0.306028  0.241046
      two    -2.820494 -1.695344

In [75]:
df.loc[("bar", "two")]

A    0.719473
B   -0.968734
C   -0.477325
Name: (bar, two), dtype: float64

In [76]:
df.loc[("bar", "two"), "A"]

np.float64(0.7194732687539579)

In [40]:
df.loc["bar"]

,A,B,C
second,,,
one,-2.574435,1.264841,0.366449
two,0.271631,-0.212004,-0.936474


### partial slicing 

In [83]:
df.loc["baz":"foo"]

A         B         C
first second                              
baz   one    -0.062280 -0.095716 -0.703227
      two     0.170354  0.790188 -0.790804
foo   one    -0.582438  0.074955  2.109549
      two     0.064365  0.434265 -1.183725

In [97]:
df.iloc[0:4]

A         B         C
first second                              
bar   one     0.783184  0.102986  0.448053
      two     1.163358 -0.496035 -0.837275
baz   one    -0.062280 -0.095716 -0.703227
      two     0.170354  0.790188 -0.790804

slicing with tuples 

In [84]:
df.loc[("baz", "two"):("qux", "one")]

A         B         C
first second                              
baz   two     0.170354  0.790188 -0.790804
foo   one    -0.582438  0.074955  2.109549
      two     0.064365  0.434265 -1.183725
qux   one     0.398612 -0.271885  1.034425

###  Slicing (extra)

- Design for slicing by index usaully doesn't include the spcified number
- However, calling with the specific index num, it includes the data (For practical usage)

In [78]:
s = pd.Series(np.random.randn(6), index=list("abcdef"))
s

a    0.354913
b   -1.517539
c    1.770469
d    0.005938
e    0.393118
f   -0.231880
dtype: float64

In [91]:
s.iloc[:3]
s.loc[:'d']

a    0.354913
b   -1.517539
c    1.770469
d    0.005938
dtype: float64

In [87]:
s[:"d"]

a   -0.902459
b   -0.436934
c   -0.063275
d   -0.303905
dtype: float64

This is most definitely a “practicality beats purity” sort of thing, but it is something to watch out for if you expect label-based slicing to behave exactly in the way that standard Python integer slicing works.

In [88]:
!pwd

'pwd'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
